# Ingest Circuits File

1. Read the file using spark dataframe reader API
2. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
3. Write to bronze delta table

In [0]:
%run ../00-Common/01.environment-config

In [0]:
%run ../00-Common/02.bronze-helpers

### Step 1 - Read the CSV file using the dataframe reader API

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
circuits_schema = StructType([
    StructField('cicrcuitId', StringType()),
    StructField('url', StringType()),
    StructField('cicrcuitName', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType())
])

In [0]:
circuits_df = (
        spark.read.format('csv')
        .option('header', 'true')
        # .option('inferSchema','true')
        .option('mode','FAILFAST')
        .schema(circuits_schema)
        .load(f"{landing_folder_path}/circuits.csv")
    )


### Step 2 - Add Metadata Columns

1. Source File
2. Ingestion Timestamp

In [0]:
circuits_final_df = add_ingestion_metadata(circuits_df)


### Step 3 - Write to bronze delta table

In [0]:
(
    circuits_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(f'{catalog_name}.{bronze_schema}.circuits')
)

In [0]:
# %sql
# SELECT * FROM formula1.bronze.circuits

In [0]:
# display(spark.table('formula1.bronze.circuits'))

In [0]:
%sql
-- DESCRIBE EXTENDED formula1.bronze.circuits